In [ ]:
import pandas as pd
import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
import networkx as nx
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import sys
sys.path.append
('../code/')
import pyspace
import libpysal as lps
from scipy.spatial import cKDTree
from libpysal.weights.distance import get_points_array
from esda import fdr
from importlib import reload
pd.set_option('display.max_rows', 500)
reload(pyspace)
import seaborn as sns
from esda.moran import Moran
# sns.set_theme(font = 'Helvetica')
%matplotlib inline
from numba import NumbaDeprecationWarning
from matplotlib.patheffects import withStroke
import pyogrio
import warnings
import esda
# Suppress NumbaDeprecationWarning
warnings.filterwarnings("ignore", category=NumbaDeprecationWarning)

In [ ]:
# data_folder = Path('../data')
data_folder  = Path('../../SanteIntegra/Data/')
results_folder = Path('../results/Getis')

In [ ]:
df_open = pd.read_parquet(data_folder/'processed'/'df_treated_open.parquet.gzip')
df_5years = pd.read_parquet(data_folder/'processed/df_treated_5years.parquet.gzip')

df_open = gpd.GeoDataFrame(df_open, crs = 4326, geometry=gpd.points_from_xy(df_open.lon_masked, df_open.lat_masked))

df_open = df_open.to_crs(2056)
df_open['E'], df_open['N'] = df_open['geometry'].x, df_open['geometry'].y

In [ ]:
# df_aos_costs = data_final[data_final.PRESTATIONS_BRUTES_AOS > data_final.MTFRANCHISECOUV] #Obsolete - revision 06.2025
df_aos_costs = df_open[df_open.PRESTATIONS_BRUTES_AOS > 0]
df_lca_costs = df_open[df_open.PRESTATIONS_BRUTES_LCA > 0]
df_cam_costs = df_open[df_open.PRESTATIONS_BRUTES_CAM > 0]

In [ ]:
df_open_2017 = df_open[df_open.NOANNEE == 2017]
df_open_2018 = df_open[df_open.NOANNEE == 2018]
df_open_2019 = df_open[df_open.NOANNEE == 2019]
df_open_2020 = df_open[df_open.NOANNEE == 2020]
df_open_2021 = df_open[df_open.NOANNEE == 2021]

In [ ]:
df_aos_costs_2017 = df_aos_costs[df_aos_costs.NOANNEE == 2017]
df_aos_costs_2018 = df_aos_costs[df_aos_costs.NOANNEE == 2018]
df_aos_costs_2019 = df_aos_costs[df_aos_costs.NOANNEE == 2019]
df_aos_costs_2020 = df_aos_costs[df_aos_costs.NOANNEE == 2020]
df_aos_costs_2021 = df_aos_costs[df_aos_costs.NOANNEE == 2021]

df_lca_costs_2017 = df_lca_costs[df_lca_costs.NOANNEE == 2017]
df_lca_costs_2018 = df_lca_costs[df_lca_costs.NOANNEE == 2018]
df_lca_costs_2019 = df_lca_costs[df_lca_costs.NOANNEE == 2019]
df_lca_costs_2020 = df_lca_costs[df_lca_costs.NOANNEE == 2020]
df_lca_costs_2021 = df_lca_costs[df_lca_costs.NOANNEE == 2021]

df_cam_costs_2017 = df_cam_costs[df_cam_costs.NOANNEE == 2017]
df_cam_costs_2018 = df_cam_costs[df_cam_costs.NOANNEE == 2018]
df_cam_costs_2019 = df_cam_costs[df_cam_costs.NOANNEE == 2019]
df_cam_costs_2020 = df_cam_costs[df_cam_costs.NOANNEE == 2020]
df_cam_costs_2021 = df_cam_costs[df_cam_costs.NOANNEE == 2021]

In [ ]:
cantons = gpd.read_file(data_folder/'raw/OFS/swissBOUNDARIES3D_1_5_LV95_LN02.gpkg', layer = 'tlm_kantonsgebiet')
communes = gpd.read_file(data_folder/'raw/OFS/swissBOUNDARIES3D_1_5_LV95_LN02.gpkg', layer = 'tlm_hoheitsgebiet')
gdf_names = communes[communes.name.isin(['Lausanne','Genève','Zürich','Basel','Bern'])]
gdf_names = gdf_names[gdf_names.einwohnerzahl.isnull()==False]

# Spatial analyses
## Data as individual points
### Global autocorrelation of main features

- AOS Yearly Spending
- LCA Yearly Spending

In [ ]:
individual_results_folder = results_folder/'Individual'

In [ ]:
globalautocorr_result_folder = individual_results_folder/'Global Autocorrelation'
if not os.path.exists(globalautocorr_result_folder):
    os.makedirs(globalautocorr_result_folder)

In [ ]:
def GlobalMoranI(db, col, year, distance, w, result_folder, seed=12345):
    xlabel = f"Global Moran's I - {col} - {year} - {distance}NN"
    file_path = result_folder / f'{xlabel}.pdf'

    # Check if the file already exists
    if file_path.exists():
        print(f"File '{file_path}' already exists. Skipping execution.")
        return None
    
    # Compute Moran's I
    y = db[col]
    np.random.seed(seed)
    mi = esda.moran.Moran(y, w)
    print(col, year, distance, mi.I, mi.p_sim, mi.z_sim)
    
    # Moran's I plot
    sns.kdeplot(mi.sim, fill=True)
    plt.vlines(mi.I, 0, plt.ylim()[1]*0.1, color='r', label='Moran\'s I')
    plt.vlines(mi.EI, 0, plt.ylim()[1]*0.1, label='Expected I')
    plt.xlabel(xlabel)
    plt.ylabel('Density')
    plt.title('Moran\'s I Distribution')
    plt.legend()
    # Save figure
    plt.savefig(file_path, dpi=320, bbox_inches='tight')
    plt.close()  # Close the plot after saving

    return mi

In [ ]:
col_names = ['ihs_cost_aos', 'ihs_cost_lca', 'ihs_cost_cam']  

# Calculate weights once for each year and store them
years = [2017, 2018, 2019, 2020, 2021]
weights_by_year = {}
nn = 32

for year, df in zip(years, [df_open_2017, df_open_2018, df_open_2019, df_open_2020, df_open_2021]):
    # Calculate weights here (w)
    # Store the weights in the dictionary
    if year not in weights_by_year.keys():
        weights_by_year[year] = lps.weights.KNN(cKDTree(get_points_array(df.geometry.centroid)), nn)


# Now iterate over each column and use the pre-calculated weights
for col_name in col_names:
    for year, df in zip(years, [df_open_2017, df_open_2018, df_open_2019, df_open_2020, df_open_2021]):
        w = weights_by_year[year]  # Retrieve pre-calculated weights
        mi = GlobalMoranI(db=df, col=col_name, year=year, distance=nn, w=w, result_folder=globalautocorr_result_folder)

### Local autocorrelation using Getis Ord Gi* statistic

In [ ]:
localautocorr_result_folder = individual_results_folder/'Local Autocorrelation'
if not os.path.exists(localautocorr_result_folder):
    os.makedirs(localautocorr_result_folder)

In [ ]:
reload(pyspace)

In [ ]:
# Now iterate over each column and use the pre-calculated weights
for col_name in col_names:
    for year, df in zip(years, [df_open_2017, df_open_2018, df_open_2019, df_open_2020, df_open_2021]):
        w = weights_by_year[year]  # Retrieve pre-calculated weights
        getis_values = pyspace.compute_getis(df, col_name, w, 999, transform_type='B', p_001=False)
        fig, ax = pyspace.plotGetisMap(df, f"{col_name}_G_cl", markersize_s=0.1, markersize_l=1, p_001=False, commune_name=False)
        xlabel = f"Getis - {col_name} - {year} - {nn}NN"
        for x, y, label in zip(gdf_names.geometry.centroid.x, gdf_names.geometry.centroid.y, gdf_names['name']):
            ax.text(x, y, label, fontsize=8, ha='right', va='bottom',
                    path_effects=[withStroke(linewidth=3, foreground='white')], zorder=8)
        file_path = localautocorr_result_folder / f'{xlabel}.png'
        plt.savefig(file_path, dpi=400, bbox_inches='tight')

In [ ]:
dict_labels = {'PRESTATIONS_BRUTES_AOS':'CM (MHI) expenditures (CHF)',
              'PRESTATIONS_BRUTES_LCA':'CAM (SI) expenditures (CHF)',
              'PRESTATIONS_BRUTES_CAM':'CAM (MHI) expenditures (CHF)',
              'ihs_cost_lca':'CAM (SI) expenditures (IHS transformed)'}

### AOS USE

In [ ]:
col_name = 'ihs_cost_aos'
nn=32
year = 2021
df = df_aos_costs_2021[~df_aos_costs_2021[col_name].isnull()]

w = lps.weights.KNN(cKDTree(get_points_array(df.geometry.centroid)), nn)
getis_values = pyspace.compute_getis(df, col_name, w, 999, transform_type='B', p_001=False)

fig, ax = pyspace.plotGetisMap(df, f"{col_name}_G_cl", markersize_s=0.08, markersize_l=1, p_001=False, commune_name=False)
for x, y, label in zip(gdf_names.geometry.centroid.x, gdf_names.geometry.centroid.y, gdf_names['name']):
    ax.text(x, y, label, fontsize=8, ha='right', va='bottom',
            path_effects=[withStroke(linewidth=3, foreground='white')], zorder=8)
# Add panel letter
fig.text(0.15, 0.25, 'A', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')
# Italicize P value
legend = ax.get_legend()
for text in legend.get_texts():
    current_text = text.get_text()
    # Replace 'P' with italicized P using LaTeX
    new_text = current_text.replace('P', '$P$')
    text.set_text(new_text)
    
xlabel = f"Getis - {col_name} - {year} - {nn}NN - positive expenses"
file_path = localautocorr_result_folder / f'{xlabel}.png'
plt.savefig(file_path, dpi=360, bbox_inches='tight')
plt.savefig('../results/Figure 1A.png', dpi=640, bbox_inches='tight')

In [ ]:
xlabel = f"Getis Bar Plot - {col_name} - {nn}NN"
file_path = Path(localautocorr_result_folder) / f'{xlabel}.png'
w.transform = 'r'
df['PRESTATIONS_BRUTES_AOS_lag'] = lps.weights.lag_spatial(w, df['PRESTATIONS_BRUTES_AOS'])
fig, ax = pyspace.plot_getis_by_class(df = df,x = f'{col_name}_G_cl',y = 'PRESTATIONS_BRUTES_AOS_lag', label = 'Annual CM (MHI) expenditures (CHF)', xtick_size=8, title_size=12, xlabel_size=8,ylabel_size= 8, p_001=False, showfliers = False)
fig.text(0.05, 0.02, 'A', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')
current_labels = [label.get_text() for label in ax.get_xticklabels()]
new_labels = [label.replace('P', '$P$') for label in current_labels]
ax.set_xticklabels(new_labels)
plt.savefig(file_path, dpi=360, bbox_inches='tight')
plt.savefig('../results/Figure S3A.png', dpi=360, bbox_inches='tight')

### CAM - MHI USE

In [ ]:
col_name = 'ihs_cost_cam'
nn=32
df = df_cam_costs_2021[~df_cam_costs_2021[col_name].isnull()]

w = lps.weights.KNN(cKDTree(get_points_array(df.geometry.centroid)), nn)
getis_values = pyspace.compute_getis(df, col_name, w, 999, transform_type='B', p_001=False)
fig, ax = pyspace.plotGetisMap(df, f"{col_name}_G_cl", markersize_s=0.08, markersize_l=1, p_001=False, commune_name=False)
for x, y, label in zip(gdf_names.geometry.centroid.x, gdf_names.geometry.centroid.y, gdf_names['name']):
    ax.text(x, y, label, fontsize=8, ha='right', va='bottom',
            path_effects=[withStroke(linewidth=3, foreground='white')], zorder=8)
# Add panel letter
fig.text(0.15, 0.25, 'B', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')
xlabel = f"Getis - {col_name} - {year} - {nn}NN - positive expenses"
legend = ax.get_legend()
for text in legend.get_texts():
    current_text = text.get_text()
    # Replace 'P' with italicized P using LaTeX
    new_text = current_text.replace('P', '$P$')
    text.set_text(new_text)
file_path = localautocorr_result_folder / f'{xlabel}.png'
plt.savefig(file_path, dpi=640, bbox_inches='tight')
plt.savefig('../results/Figure 1B.png', dpi=640, bbox_inches='tight')

In [ ]:
xlabel = f"Getis Bar Plot - {col_name} - {nn}NN - positive expenses"
file_path = Path(localautocorr_result_folder) / f'{xlabel}.png'
w.transform = 'r'
df['PRESTATIONS_BRUTES_CAM_lag'] = lps.weights.lag_spatial(w, df['PRESTATIONS_BRUTES_CAM'])
fig, ax = pyspace.plot_getis_by_class(df = df,x = f'{col_name}_G_cl',y = 'PRESTATIONS_BRUTES_CAM_lag', label = 'Annual CAM (MHI) expenditures (CHF)', xtick_size=8, title_size=12, xlabel_size=8,ylabel_size= 8, p_001=False, showfliers = False)
current_labels = [label.get_text() for label in ax.get_xticklabels()]
new_labels = [label.replace('P', '$P$') for label in current_labels]
ax.set_xticklabels(new_labels)
fig.text(0.05, 0.02, 'B', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')
# Italicize P value
legend = ax.get_legend()
for text in legend.get_texts():
    current_text = text.get_text()
    # Replace 'P' with italicized P using LaTeX
    new_text = current_text.replace('P', '$P$')
    text.set_text(new_text)

    
plt.savefig(file_path, dpi=320, bbox_inches='tight')
plt.savefig('../results/Figure S3B.png', dpi=300, bbox_inches='tight')

## LCA USE

In [ ]:
col_name = 'ihs_cost_lca'
nn=32
df = df_lca_costs_2021[~df_lca_costs_2021[col_name].isnull()]

w = lps.weights.KNN(cKDTree(get_points_array(df.geometry.centroid)), nn)
getis_values = pyspace.compute_getis(df, col_name, w, 999, transform_type='B', p_001=False)
fig, ax = pyspace.plotGetisMap(df, f"{col_name}_G_cl", markersize_s=0.08, markersize_l=1, p_001=False, commune_name=False)

for x, y, label in zip(gdf_names.geometry.centroid.x, gdf_names.geometry.centroid.y, gdf_names['name']):
    ax.text(x, y, label, fontsize=8, ha='right', va='bottom',
            path_effects=[withStroke(linewidth=3, foreground='white')], zorder=8)
# ax.set_title('A', loc = 'left', size= 16)
ax.set_axis_off()  # Hide axes
legend = ax.get_legend()
for text in legend.get_texts():
    current_text = text.get_text()
    # Replace 'P' with italicized P using LaTeX
    new_text = current_text.replace('P', '$P$')
    text.set_text(new_text)
fig.text(0.15, 0.25, 'C', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')
xlabel = f"Getis - {col_name} - {year} - {nn}NN - positive expenses"
file_path = localautocorr_result_folder / f'{xlabel}.png'
plt.savefig(file_path, dpi=640, bbox_inches='tight')
plt.savefig('../results/Figure 1C.png', dpi=640, bbox_inches='tight')

In [ ]:
xlabel = f"Getis Bar Plot - {col_name} - {nn}NN - positive expenses"
file_path = Path(localautocorr_result_folder) / f'{xlabel}.png'
w.transform = 'r'
df['PRESTATIONS_BRUTES_LCA_lag'] = lps.weights.lag_spatial(w, df['PRESTATIONS_BRUTES_LCA'])
fig, ax = pyspace.plot_getis_by_class(df = df,x = f'{col_name}_G_cl',y = 'PRESTATIONS_BRUTES_LCA_lag', label = 'Annual CAM (SI) expenditures (CHF)', xtick_size=8, title_size=12, xlabel_size=8,ylabel_size= 8, p_001=False, showfliers = False)
current_labels = [label.get_text() for label in ax.get_xticklabels()]
new_labels = [label.replace('P', '$P$') for label in current_labels]
ax.set_xticklabels(new_labels)
fig.text(0.05, 0.02, 'C', fontsize=16, fontweight='bold', 
         transform=fig.transFigure, ha='left', va='bottom')
plt.savefig(file_path, dpi=320, bbox_inches='tight')
plt.savefig('../results/Figure S3C.png', dpi=300, bbox_inches='tight')